# %% [markdown]
# # Used Car Listings — Cleaning Notebook (tsaustin)
# Source: data/clean_listings.csv
# Output: data/clean_listings_clean.{csv,parquet}
# This notebook:
# - Loads the CSV
# - Fixes types and text
# - Filters obvious outliers
# - Deduplicates (VIN if present, else make/model/year/mileage)
# - Adds a few helper features
# - Saves cleaned files for the dashboard/model

In [13]:
import os, re, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import os
from pathlib import Path

# Always run relative to repo root
os.chdir("/workspaces/used-car-price-app")

SRC = Path("data/clean_listings.csv")
assert SRC.exists(), f"Missing source file: {SRC}"

In [14]:
SRC = Path("/workspaces/used-car-price-app/data/clean_listings.csv")
OUT_CSV = Path("data/clean_listings_clean.csv")
OUT_PARQ = Path("data/clean_listings_clean.parquet")



In [15]:
assert SRC.exists(), f"Missing source file: {SRC}"

# %%
# Load
df = pd.read_csv(SRC)
print("Raw shape:", df.shape)
df.head(3)

# %%
# Inspect columns
df.columns.tolist()

# %%
# --- Canonical column names we care about
rename_map = {
    "pricesold":"price",
    "saleprice":"price",
    "list_price":"price",
    "miles":"mileage",
    "odometer":"mileage",
    "brand":"make",
    "manufacturer":"make",
    "car_model":"model",
    "series":"model",
    "trim":"model",  # keep best-effort
    "model_year":"year",
}
df = df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns})


Raw shape: (70179, 13)


In [16]:
need = ["price","year","mileage","make","model"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise ValueError(f"Required columns missing: {missing}")

# %%
# Type coercions
for c in ["price","year","mileage"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

for c in ["make","model"]:
    df[c] = df[c].astype(str).str.strip()

# %%
# Text normalization
def clean_make(s: pd.Series) -> pd.Series:
    return s.str.title().str.replace(r"\s+", " ", regex=True).str.strip()

def clean_model(s: pd.Series) -> pd.Series:
    s = s.str.replace(r"[^A-Za-z0-9\-\s]", "", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

df["make"] = clean_make(df["make"])
df["model"] = clean_model(df["model"])

In [17]:
RANGE = {
    "price":   (1000, 250_000),
    "year":    (1990, 2026),
    "mileage": (0, 300_000),
}
for c,(lo,hi) in RANGE.items():
    df = df[df[c].between(lo, hi, inclusive="both")]

# Drop rows missing essentials after cleaning
df = df.dropna(subset=["price","year","mileage","make","model"])

# Cast to int where appropriate
df["year"] = df["year"].round().astype(int)
df["mileage"] = df["mileage"].round().astype(int)

print("After basic cleaning:", df.shape)

# %%
# Deduping
keys = []
if "vin" in df.columns:
    keys = ["vin"]
else:
    for k in ["make","model","year","mileage","state","dealer"]:
        if k in df.columns:
            keys.append(k)

if keys:
    before = len(df)
    df = df.drop_duplicates(subset=keys, keep="first")
    print(f"Deduped {before - len(df)} rows using keys: {keys}")
else:
    print("No dedupe keys found; skipped de-duplication")

# %%
# Feature engineering (useful for model & dashboard enrich)
if "state" in df:  # normalize state text a bit
    df["state"] = df["state"].astype(str).str.upper().str.strip()

current_year = 2025
df["age"] = current_year - df["year"]

# Price per mile (safe guard div by 0)
df["price_per_mile"] = df["price"] / df["mileage"].replace({0: np.nan})
df["price_per_mile"] = df["price_per_mile"].replace([np.inf, -np.inf], np.nan)

# Log price (commonly helpful)
df["log_price"] = np.log1p(df["price"])

df.head(5)

After basic cleaning: (70179, 13)
Deduped 880 rows using keys: ['make', 'model', 'year', 'mileage']


,ID,price,yearsold,zipcode,mileage,make,model,year,Trim,Engine,BodyType,NumCylinders,DriveType,age,price_per_mile,log_price
0,119660,8750,2020,33449,55000,Jaguar,XJS,1995,2+2 Cabriolet,4.0L In-Line 6 Cylinder,Convertible,6,RWD,30,0.159091,9.076923
1,64287,44000,2019,07728,40703,Porsche,911,2002,Turbo X-50,3.6L,Coupe,6,AWD,23,1.081001,10.691968
2,5250,70000,2019,07627,6500,Land Rover,Defender,1997,NaN,4.0 Liter Fuel Injected V8,NaN,0,4WD,28,10.769231,11.156265
3,29023,1330,2019,07043,167000,Honda,Civic,2001,EX,NaN,Coupe,4,FWD,24,0.007964,7.193686
4,158271,20000,2020,333**,51674,Jeep,Wrangler,2015,SPORT,3.6L Flexible V6,SUV,6,4WD,10,0.387042,9.903538


In [18]:
summary = {
    "rows": len(df),
    "columns": list(df.columns),
    "price_mean": float(df["price"].mean()),
    "price_p50": float(df["price"].median()),
    "mileage_mean": float(df["mileage"].mean()),
    "year_min": int(df["year"].min()),
    "year_max": int(df["year"].max()),
}
summary

{'rows': 69299,
 'columns': ['ID',
  'price',
  'yearsold',
  'zipcode',
  'mileage',
  'make',
  'model',
  'year',
  'Trim',
  'Engine',
  'BodyType',
  'NumCylinders',
  'DriveType',
  'age',
  'price_per_mile',
  'log_price'],
 'price_mean': 11451.862653140737,
 'price_p50': 6700.0,
 'mileage_mean': 102772.20261475634,
 'year_min': 1990,
 'year_max': 2020}

In [19]:
OUT_PARQ.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUT_PARQ, index=False)
df.to_csv(OUT_CSV, index=False)

print(f"Saved:\n - {OUT_PARQ}\n - {OUT_CSV}\nFinal shape: {df.shape}")


Saved:
 - data/clean_listings_clean.parquet
 - data/clean_listings_clean.csv
Final shape: (69299, 16)


In [20]:
df.head(5000).to_csv("data/clean_listings_clean_preview5k.csv", index=False)
print("Also wrote: data/clean_listings_clean_preview5k.csv")


Also wrote: data/clean_listings_clean_preview5k.csv
